# tpu-retrieval-bench on Colab TPU

Runtime > Change runtime type > TPU v5e-1. Results land in `results/` and are zipped at the end.

In [ ]:
import sys
print(sys.version)
if sys.version_info < (3, 12):
    print('WARNING: python < 3.12. Fallback: pip install "jax[tpu]<0.11" and '
          'pip install --ignore-requires-python -e ".[data,plot]" (repo declares >= 3.12).')

In [ ]:
!git clone https://github.com/maxpetrusenko/tpu-retrieval-bench
%cd tpu-retrieval-bench
!pip install -q -e ".[data,plot]" "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

In [ ]:
import jax
dev = jax.devices()[0]
assert dev.platform == "tpu", f"not a TPU runtime: {jax.devices()}"
print(dev.device_kind, len(jax.devices()), "device(s)")

## Synthetic sweep

In [ ]:
!python bench/run.py --algos jit_topk,blockwise,pallas_score,approx_max_k --m 1,16,128,1024 --n 65536,1000000 --d 128 --k 10 --dtype bf16

## ann-benchmarks SIFT (525 MB)

In [ ]:
!python bench/download.py sift

In [ ]:
# Dataset run. If bench/run.py in your checkout has no dataset flag yet, evaluate recall directly:
import numpy as np
from trbench import datasets
from trbench.jit_topk import make_jit_topk
ann = datasets.load_ann("sift-128-euclidean", n_test=1000)
f = make_jit_topk(10, metric=ann["metric"])
scores, ids = f(jax.device_put(ann["test"]), jax.device_put(ann["train"]))
dists = datasets.scores_to_distances(np.asarray(scores), ann["metric"], ann["test"])
print("knn recall@10", datasets.knn_recall(dists, ann["distances"], 10))
print("id-set recall@10", datasets.id_set_recall(np.asarray(ids), ann["neighbors"], 10))

## Profiler trace for one config

In [ ]:
import jax.numpy as jnp
from trbench.jit_topk import make_jit_topk
rng = np.random.default_rng(0)
q = jnp.asarray(rng.standard_normal((128, 128), dtype=np.float32))
e = jnp.asarray(rng.standard_normal((1_000_000, 128), dtype=np.float32))
f = make_jit_topk(10)
jax.block_until_ready(f(q, e))  # warm
with jax.profiler.trace("results/profile_jit_topk_m128_n1e6_d128"):
    jax.block_until_ready(f(q, e))

## Plots and download

In [ ]:
!python bench/plot.py

In [ ]:
!zip -qr results.zip results docs/img
from google.colab import files
files.download("results.zip")